# Exercise 05 — continuous streaming + anomalies

**Goal:** simulate a realistic GPS feed from the riders' phones: continuous, irregular, occasionally noisy. This is the producer side of the canonical pattern "devices push → backend aggregates → alerts fire".

**How to use this notebook:**

1. In a separate tab, open [`exercise_02_consume_basic.ipynb`](exercise_02_consume_basic.ipynb), run *Step 1* to *Step 3* and then the *Live streaming* cell — it will sit and wait for messages.
2. Run the streaming producer below. Events should appear live in the consumer.
3. Stop the producer with the **■** button when you're done.

The setup cell prepares four riders and a helper `move(rider)` that moves a rider a few metres and returns the new position and speed.

In [ ]:
from confluent_kafka import Producer
from datetime import datetime
import json, time, random

BROKER = 'redpanda:29092'

producer = Producer({
    'bootstrap.servers': BROKER,
    'client.id':         'streaming-producer',
})

riders = ['rider_3', 'rider_5', 'rider_6', 'rider_7']
NORMAL_MAX_KMH = 35.0   # a bike or scooter in town does not go faster

# Every rider starts somewhere near Lucerne main station.
positions = {r: [47.0502 + random.uniform(-0.01, 0.01),
                 8.3093 + random.uniform(-0.01, 0.01)] for r in riders}


def move(rider):
    """Move the rider a few metres and return (lat, lon, speed_kmh)."""
    positions[rider][0] += random.uniform(-0.0003, 0.0003)
    positions[rider][1] += random.uniform(-0.0003, 0.0003)
    speed = round(random.uniform(0.0, NORMAL_MAX_KMH), 1)   # 0 = waiting at a restaurant
    return round(positions[rider][0], 5), round(positions[rider][1], 5), speed

print('Ready.')

## Step 1 — continuous stream (1 event every 0.5–2 s)

**What's interesting here:** a real producer never ends — each rider's phone keeps sending pings as long as the shift lasts. We let `time.sleep` create a realistic *event rate*; here we simulate all riders from one process. So the notebook does not run forever by accident, it stops after `MAX_EVENTS` pings (about 5 minutes) or when you press ■.

**Task — fill in the event dict.** Use the names `rider_id`, `lat`, `lon`, `speed_kmh`, `timestamp`.

In [ ]:
MAX_EVENTS = 250
count = 0
print(f'{"Time":>10} | {"#":>4} | {"Rider":>7} | {"Lat":>9} | {"Lon":>8} | {"km/h":>5}')
print('-' * 60)

try:
    while count < MAX_EVENTS:
        rider           = random.choice(riders)
        lat, lon, speed = move(rider)

        event = json.dumps({
            # TODO: fill in the event fields
            #   'rider_id', 'lat', 'lon', 'speed_kmh', 'timestamp'
        })

        producer.produce('rider-locations', key=rider.encode(), value=event.encode())
        producer.flush()

        count += 1
        ts = datetime.now().strftime('%H:%M:%S')
        print(f'{ts:>10} | {count:>4} | {rider:>7} | {lat:>9.5f} | {lon:>8.5f} | {speed:>5.1f}')
        time.sleep(random.uniform(0.5, 2.0))

except KeyboardInterrupt:
    print(f'\nStopped. Total events sent: {count}')

## Task A — burst mode

Send **50 pings as fast as possible** (no `sleep`).

**What to observe:**

- The producer barely sweats — `produce()` only enqueues, the background thread sends asynchronously.
- One `flush()` at the *end* is enough; you don't need one per event (and shouldn't, for performance).
- The consumer lags only briefly: Kafka was designed for exactly this kind of bursty traffic — think of the Friday-evening rush when hundreds of riders are on the road.

In [ ]:
# TODO: send 50 pings as fast as possible, then producer.flush() once at the end

## Task B — anomaly simulation

Real GPS streams contain **outliers**. A phone loses its satellite fix in a tunnel or between tall buildings, and the next fix places the rider hundreds of metres away. The app then computes an **impossible speed**: no bike or scooter in town does 80 km/h or more. You can't ignore such glitches: they corrupt the average speed, make the rider's icon jump across the map and spoil the delivery-time estimate unless they are flagged.

**Task:** send 30 pings where ~20 % are GPS glitches (`speed_kmh` ≥ 3× `NORMAL_MAX_KMH`, i.e. 105 km/h or more). Add an `'anomaly': True` flag in the JSON.

**Why ship the flag along with the data?** Two reasons:

1. *Lineage* — downstream you can audit how many anomalies were in the original stream.
2. *Self-describing data* — the consumer doesn't need to repeat the threshold logic.

After running, find one of your anomalous messages in Redpanda Console (search the *Messages* tab for `"anomaly": true`).

In [ ]:
# TODO: send 30 pings; ~20% should have speed_kmh >= 3 * NORMAL_MAX_KMH
#       and 'anomaly': True. Print '<-- ANOMALY' next to anomalous rows.

## What you learned

- Kafka producers are non-blocking — call `produce()` in a tight loop and `flush()` only at the end.
- A *streaming* mindset: data is infinite, the program is the pipeline, not the result.
- Including an explicit `anomaly` flag is cheaper and more reliable than re-deriving it downstream.